In [ ]:
# from dotenv import load_dotenv
import os
import streamlit as st
from notion_client import Client


# load_dotenv()

# NOTION_TOKEN = os.getenv("NOTION_TOKEN")
NOTION_TOKEN = st.secrets.get("NOTION_TOKEN") or os.getenv("NOTION_TOKEN")

if not NOTION_TOKEN:
    st.error("NOTION_TOKEN is missing. Add it in Streamlit Cloud secrets.")
    st.stop()
    
if not NOTION_TOKEN:
    raise ValueError("NOTION_TOKEN not found")

client = Client(auth=NOTION_TOKEN)


MONDAY_CHECKS_DATA_SOURCE_ID = "31bcc2a6c00c80958f52000bc92cde8c"
# 31bcc2a6c00c80958f52000bc92cde8c - real
# 351cc2a6c00c80c8bd0c000b83918902 - test 
PROJECTS_DATA_SOURCE_ID = "30acc2a6c00c81179587000b85dd79c0"

PROJECT_PROPERTY_NAME = "Project"
TITLE_PROPERTY_NAME = "Name"
IS_TEMPLATE_PROPERTY_NAME = "Is Template"


def get_title_property_as_string(page: dict, prop_name: str) -> str:
    prop = page["properties"].get(prop_name)

    if not prop or prop["type"] != "title":
        return "(Untitled)"

    return "".join(t.get("plain_text", "") for t in prop["title"]) or "(Untitled)"


def get_checkbox_value(page: dict, prop_name: str) -> bool:
    prop = page["properties"].get(prop_name)

    if not prop or prop["type"] != "checkbox":
        return False

    return prop["checkbox"]


def get_project_page_id(project_name: str) -> str | None:
    query = client.data_sources.query(
        data_source_id=PROJECTS_DATA_SOURCE_ID,
        page_size=1,
        filter={
            "property": TITLE_PROPERTY_NAME,
            "title": {"equals": project_name},
        },
    )

    results = query["results"]

    if not results:
        return None

    return results[0]["id"]


def query_template_rows_for_project(project_page_id: str) -> list[dict]:
    rows = []
    start_cursor = None

    while True:
        query = client.data_sources.query(
            data_source_id=MONDAY_CHECKS_DATA_SOURCE_ID,
            page_size=100,
            start_cursor=start_cursor,
            filter={
                "and": [
                    {
                        "property": PROJECT_PROPERTY_NAME,
                        "relation": {
                            "contains": project_page_id,
                        },
                    },
                    {
                        "property": IS_TEMPLATE_PROPERTY_NAME,
                        "checkbox": {
                            "equals": True,
                        },
                    },
                ]
            },
        )

        rows.extend(query["results"])

        if not query.get("has_more"):
            break

        start_cursor = query.get("next_cursor")

    return rows


def main() -> None:
    project_name = input("Which Project should template rows be deleted for? ").strip()

    if not project_name:
        print("No project entered. Exiting.")
        return

    project_page_id = get_project_page_id(project_name)

    if not project_page_id:
        print(f'No project found named "{project_name}".')
        return

    rows = query_template_rows_for_project(project_page_id)

    if not rows:
        print(f'No checked template rows found for project "{project_name}".')
        return

    print(f'Found {len(rows)} checked template row(s) for project "{project_name}".')

    trashed_count = 0

    for row in rows:
        title = get_title_property_as_string(row, TITLE_PROPERTY_NAME)

        if row.get("in_trash") or row.get("archived"):
            print(f'Skipping "{title}" — already trashed.')
            continue

        client.pages.update(page_id=row["id"], in_trash=True)

        print(f'Trashed template row: "{title}"')
        trashed_count += 1

    print()
    print(f"Done. Trashed {trashed_count} template row(s).")


if __name__ == "__main__":
    main()

No project found named "asdgsdgsdfh".
